In [38]:
import pandas as pd
import numpy as np
import json
import plotnine as p9

from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.feature_selection import mutual_info_regression
from sklearn.base import BaseEstimator, TransformerMixin

In [39]:
IS_FORWARD_STEPWISE_COMPLETE_FEATURE_SELECTION_RUN = True
IS_FORWARD_STEPWISE_EPISODE_LENGTH_IMPUTE_MODEL_FEATURE_SELECTION_RUN = False

# Load

In [40]:
podcasts_train = pd.read_csv("./data/external/train.csv")
podcasts_train.columns = [x.lower() for x in podcasts_train.columns]
podcasts_train = podcasts_train.assign(
    podcast_name_episode_title = lambda df_: df_['podcast_name'] + '|' + df_['episode_title'],
    podcast_name_episode_title_day = lambda df_: df_['podcast_name'] + '|' + df_['episode_title'] + '|' + df_['publication_day'],
    podcast_name_episode_title_day_length = lambda df_: df_['podcast_name'] + '|' + df_['episode_title'] + '|' + df_['publication_day'] + '|' + df_['episode_length_minutes'].astype(str),
)

podcasts_train = podcasts_train.assign(
    episode_count = lambda df_: df_['episode_title'].str.extract(r'Episode (\d+)').astype(int)
)

# surprisingly, one record has finer grain than episode-length-releaseDay.
# odd, but de-dupe over all columns yields no drops.
# ASSUME: given grain of data is appropriate; 
# without further information, unclear how to action on possible dupes
assert not podcasts_train['podcast_name_episode_title_day_length'].is_unique
assert podcasts_train.drop_duplicates().shape[0] == podcasts_train.shape[0]

In [ ]:
podcasts_train.head()

In [ ]:
podcasts_train['episode_count'].describe()

In [ ]:
podcasts_train.shape

In [ ]:
podcasts_train.isnull().mean()

In [ ]:
podcasts_train['listening_time_minutes'].std()

In [46]:
def transform_categorical_interactions(X):

    X = X.assign(
        podcast_name_publication_day = lambda df_: df_['podcast_name'] + '_' + df_['publication_day'],
        podcast_name_publication_time = lambda df_: df_['podcast_name'] + '_' + df_['publication_time'],
        podcast_name_publication_day_publication_time = lambda df_: df_['podcast_name'] + '_' + df_['publication_day'] + df_['publication_time'],
        podcast_name_episode_sentiment = lambda df_: df_['podcast_name'] + '_' + df_['episode_sentiment'],

        genre_publication_day = lambda df_: df_['genre'] + '_' + df_['publication_day'],
        genre_publication_time = lambda df_: df_['genre'] + '_' + df_['publication_time'],
        genre_episode_sentiment = lambda df_: df_['genre'] + '_' + df_['episode_sentiment'],

        publication_day_publication_time = lambda df_: df_['publication_day'] + '_' + df_['publication_time'],
        publication_day_episode_sentiment = lambda df_: df_['publication_day'] + '_' + df_['episode_sentiment'],
        publication_time_episode_sentiment = lambda df_: df_['publication_time'] + '_' + df_['episode_sentiment'],
        publication_time_publication_day_episode_sentiment = lambda df_: df_['publication_day'] + '_' + df_['publication_time'] + df_['episode_sentiment']
    )

    return X

podcasts_train = transform_categorical_interactions(podcasts_train)

In [110]:
# before jumping to features & modeling -- 
# 'Data Description Report' confirms expected types.
# downstream, subset from these to obtain _features_ for models

COLUMNS_NUMERIC = [
    # 'episode_length_minutes', 
    # 'host_popularity_percentage', 
    # 'guest_popularity_percentage', 
    # 'number_of_ads', 
    # 'episode_count'
    ]

COLUMNS_CATEGORICAL = [

    'episode_length_minutes', 
    'host_popularity_percentage', 
    'guest_popularity_percentage', 
    'number_of_ads', 
    'episode_count',

    'podcast_name', 
    'genre', 
    'publication_day', 
    'publication_time', 
    'episode_sentiment',

    'podcast_name_publication_day',
    'podcast_name_publication_time',
    'podcast_name_publication_day_publication_time',
    'podcast_name_episode_sentiment',

    'genre_publication_day',
    'genre_publication_time',
    'genre_episode_sentiment',

    'publication_day_publication_time',
    'publication_day_episode_sentiment',
    'publication_time_episode_sentiment',
    'publication_time_publication_day_episode_sentiment'
    
    ]

In [109]:
podcasts_train.head()

,id,podcast_name,episode_title,episode_length_minutes,genre,host_popularity_percentage,publication_day,publication_time,guest_popularity_percentage,number_of_ads,...,podcast_name_publication_time,podcast_name_publication_day_publication_time,podcast_name_episode_sentiment,genre_publication_day,genre_publication_time,genre_episode_sentiment,publication_day_publication_time,publication_day_episode_sentiment,publication_time_episode_sentiment,publication_time_publication_day_episode_sentiment
0,0,Mystery Matters,Episode 98,NaN,True Crime,74.81,Thursday,Night,NaN,0.0,...,Mystery Matters_Night,Mystery Matters_ThursdayNight,Mystery Matters_Positive,True Crime_Thursday,True Crime_Night,True Crime_Positive,Thursday_Night,Thursday_Positive,Night_Positive,Thursday_NightPositive
1,1,Joke Junction,Episode 26,119.80,Comedy,66.95,Saturday,Afternoon,75.95,2.0,...,Joke Junction_Afternoon,Joke Junction_SaturdayAfternoon,Joke Junction_Negative,Comedy_Saturday,Comedy_Afternoon,Comedy_Negative,Saturday_Afternoon,Saturday_Negative,Afternoon_Negative,Saturday_AfternoonNegative
2,2,Study Sessions,Episode 16,73.90,Education,69.97,Tuesday,Evening,8.97,0.0,...,Study Sessions_Evening,Study Sessions_TuesdayEvening,Study Sessions_Negative,Education_Tuesday,Education_Evening,Education_Negative,Tuesday_Evening,Tuesday_Negative,Evening_Negative,Tuesday_EveningNegative
3,3,Digital Digest,Episode 45,67.17,Technology,57.22,Monday,Morning,78.70,2.0,...,Digital Digest_Morning,Digital Digest_MondayMorning,Digital Digest_Positive,Technology_Monday,Technology_Morning,Technology_Positive,Monday_Morning,Monday_Positive,Morning_Positive,Monday_MorningPositive
4,4,Mind & Body,Episode 86,110.51,Health,80.07,Monday,Afternoon,58.68,3.0,...,Mind & Body_Afternoon,Mind & Body_MondayAfternoon,Mind & Body_Neutral,Health_Monday,Health_Afternoon,Health_Neutral,Monday_Afternoon,Monday_Neutral,Afternoon_Neutral,Monday_AfternoonNeutral


# Fit, Evaluate Baseline Model

In [48]:
features_numeric_impute = COLUMNS_NUMERIC
features_categorical = COLUMNS_CATEGORICAL

In [49]:
# feature_transform_pipeline = ColumnTransformer(
#     [
#         ("target_encode", TargetEncoder(), features_categorical),
#         ("impute", SimpleImputer(strategy='mean', add_indicator=True), features_numeric_impute)
#     ],
#     remainder='drop',
#     verbose_feature_names_out=False
# )

# pipeline_e2e = Pipeline(
#     [
#         ("transform_features", feature_transform_pipeline), 
#         ("model", RandomForestRegressor(n_jobs=-1))
#     ]
#     )

# X = podcasts_train.drop(columns=['listening_time_minutes'])
# y = podcasts_train['listening_time_minutes']

# # to accelerate runtime, prioritize parallelization of RF fit
# scores = cross_val_score(pipeline_e2e, X, y, scoring='neg_root_mean_squared_error', cv=5, verbose=2)
# scores.mean()

In [ ]:
# safe to encode onehots upstream, versus on-the-fly. no data leakage risk
# located upstream for use throughout

# omit some very high cardinality, 3-way interactions
COLUMNS_TO_ONEHOT = [
    'podcast_name', 
    'genre', 
    'publication_day', 
    'publication_time', 
    'episode_sentiment',

    'podcast_name_publication_day',
    'podcast_name_publication_time',
    'podcast_name_episode_sentiment',

    'genre_publication_day',
    'genre_publication_time',
    'genre_episode_sentiment',

    'publication_day_publication_time',
    'publication_day_episode_sentiment',
    'publication_time_episode_sentiment',
    'publication_time_publication_day_episode_sentiment'
    
    ]

onehot_encoder = OneHotEncoder(categories='auto', sparse_output=False).set_output(transform='pandas')
X_onehot = onehot_encoder.fit_transform(podcasts_train[COLUMNS_TO_ONEHOT])
features_onehot = list(X_onehot.columns)

XY = pd.concat([podcasts_train, X_onehot], axis=1)

XY.shape

# Data Exploration Report

In [ ]:
(
    podcasts_train
    .query("podcast_name == 'Digital Digest'")
    [['episode_length_minutes', 'listening_time_minutes']]
    .plot
    .scatter('episode_length_minutes', 'listening_time_minutes')
);

In [ ]:
(
    podcasts_train
    .query("podcast_name == 'Digital Digest'")
    .assign(y_normalized = lambda df_: (df_['listening_time_minutes'] / df_['episode_length_minutes']).replace(np.inf, np.nan))
    [['y_normalized']]
    .plot
    .kde()
);

In [ ]:
(
    podcasts_train
    .query("podcast_name == 'Digital Digest'")
    .assign(y_normalized = lambda df_: (df_['listening_time_minutes'] / df_['episode_length_minutes']).replace(np.inf, np.nan))
    [['y_normalized']]
    .query("y_normalized < 9")
    .plot
    .kde()
);

In [ ]:
(
    podcasts_train
    .query("podcast_name == 'Digital Digest'")
    .assign(y_normalized = lambda df_: (df_['listening_time_minutes'] / df_['episode_length_minutes']).replace(np.inf, np.nan))
    [['y_normalized']]
    .query("y_normalized < 9")
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
)

In [18]:
# hypothesis generation:
    # mixture model, is_outlier or not?
    # improved episode_length_minutes imputation -- by podcast, for example
    # outcome to model is actually, y normalized by episode length? target encoding is naive to important feature of episode length.
    # error summary by pred_bucket?

In [19]:
# considered forward-stepwise procedure excluding episode_length,
# but model performances plunged. Moreover, that feature is highly influential --
# much simpler to try and predict it ahead of time.

# XY_mi = (
#     XY
#     .query("podcast_name == 'Digital Digest'")
#     [['episode_length_minutes', 'host_popularity_percentage', 'guest_popularity_percentage', 'number_of_ads', 'episode_count']]
#     .dropna()
# )

# y = XY_mi['episode_length_minutes']
# X = XY_mi.drop(columns='episode_length_minutes')

# mutual_info_regression(X, y)

# very little correlation

In [ ]:
XY.query("podcast_name == 'Digital Digest'").plot.scatter('episode_count', 'episode_length_minutes');

In [ ]:
XY.query("podcast_name == 'Digital Digest'")['episode_length_minutes'].plot.hist();

In [ ]:
(
    p9.ggplot(XY.query("podcast_name.isin(['Brain Boost', 'Business Briefs', 'Criminal Minds', 'Game Day'])")) + 
    p9.theme_bw() + 
    p9.geom_density(p9.aes('episode_length_minutes', group='podcast_name', color='podcast_name'), adjust=2)
)

In [ ]:
XY[['podcast_name', 'episode_length_minutes']].groupby('podcast_name').agg('mean')

In [ ]:
XY['episode_length_minutes'].std()

In [ ]:
(
    p9.ggplot(XY.assign(has_null = lambda df_: df_['episode_length_minutes'].isnull().astype(int))) + 
    p9.theme_bw() + 
    p9.geom_density(p9.aes('listening_time_minutes', group='has_null', color='has_null'), adjust=3)
)

# Forward Stepwise Features Selection 

In [ ]:
def create_transformers_argument(features_model, features_universe_target_encode, features_universe_impute):

    transformers = []

    features_target_encode = [ftr for ftr in features_model if ftr in features_universe_target_encode]
    if features_target_encode:
        spec = ('target_encode', TargetEncoder(), features_target_encode)
        transformers.append(spec)

    features_impute = [ftr for ftr in features_model if ftr in features_universe_impute]
    if features_impute:
        spec = ('impute', SimpleImputer(strategy='mean', add_indicator=True), features_impute)
        transformers.append(spec)

    return transformers

## Reduce Features Universe

In [ ]:
features_universe = features_categorical + features_onehot + features_numeric_impute

transformers = create_transformers_argument(features_universe, features_categorical, features_numeric_impute)
feature_transform_pipeline = (
    ColumnTransformer(transformers, remainder='passthrough', verbose_feature_names_out=False)
    .set_output(transform='pandas')
)

XY_trfm = feature_transform_pipeline.fit_transform(XY[features_universe], XY['listening_time_minutes'])

# parallelization is critical for MI to complete, with ~1,000 features
mutual_information = mutual_info_regression(XY_trfm, XY['listening_time_minutes'], n_jobs=-1)

In [ ]:
mutual_information = (
    pd.Series(mutual_information, index=XY_trfm.columns)
    .sort_values(ascending=False)
)
# mutual_information.to_dict()

features_low_importance = list(mutual_information[-500:].index)

In [ ]:
# import joblib
# joblib.cpu_count()

## Execute Selection

In [69]:
class Float32Transformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.astype(np.float32)

In [ ]:
import time

start = time.time()

# preferable for > 200K observation datasets, vs SVR
# Chris Deotte implements LinearSVC: https://www.kaggle.com/code/cdeotte/rapids-svc-w-feature-engineering-lb-0-856
# don't expect that LinearSVR implicitly constructs interactions of features.
# will it at least run faster than Ridge?
# TODO: does GPU-based LinearSVR accelerate this procedure?
from sklearn.svm import LinearSVR

if IS_FORWARD_STEPWISE_COMPLETE_FEATURE_SELECTION_RUN:

    # for each candidate feature:
        # fit model--assume linear form, estimate via Ridge--and evaluate with out-of-fold CV average.
            # expect Ridge sufficient because it detects main effects, assumed more important than 2nd-order effects. 
    # compare models -- which model/marginal feature is best?
    # add marginal feature to champion set, then repeat exercise. until best 'challenger' model loses to champion.

    features_marginal = features_categorical + features_onehot + features_numeric_impute
    # using features universe, model search times practically infeasible
    features_marginal = [x for x in features_marginal if x not in features_low_importance]
    features_champion = []

    # dummy, to kick off procedure
    champion_score = 10_000
    challenger_score = 1_000
    champions_score_sequence = [champion_score]

    # when challenger improves upon champion, continue extending challenger.
    # when challenger loses to champion, challenger has become too complex.
    while champion_score >= challenger_score:

        challengers_scores = {}
        for feature_marginal in features_marginal:

            features_challenger = features_champion + [feature_marginal]

            # experimented with data sample to 10,000, then calculate, 
            # but that approximation was untrustworthy. Procedure finished way too early,
            # and scores were optimistic (interpreted: outliers drive score up).
            X_challenger = XY[features_challenger]
            y = XY['listening_time_minutes']

            transformers = create_transformers_argument(features_challenger, features_categorical, features_numeric_impute)
            feature_transform_pipeline = ColumnTransformer(transformers, remainder='passthrough', verbose_feature_names_out=False)

            pipeline_e2e = Pipeline(
                [
                    ("transform_features", feature_transform_pipeline), 
                    ("standard_scale", StandardScaler()),
                    # objective is to determine which features matter, faster:
                    # expect we can compromise on data precision.
                    ('to_float32', Float32Transformer()),
                    # experimented with solver='saga' for large data,
                    # but that performed slower than 'auto'
                    ("model", Ridge(alpha=0.1))
                    # ("model", LinearSVR(C=0.1))
                ]
                )
            
            scores = cross_val_score(
                pipeline_e2e, 
                X_challenger, 
                y.astype('float32'), 
                scoring='neg_root_mean_squared_error', 
                cv=5,  
                # too often, parallel workers fail mid-execution.
                # that could be explained by maximized workers "burning out".
                n_jobs=7
                )
            score_cv_summary = scores.mean()

            challengers_scores[feature_marginal] = -1 * score_cv_summary

        feature_marginal_challenger = min(challengers_scores, key=challengers_scores.get)
        challenger_score = challengers_scores[feature_marginal_challenger]

        print(f"Challenger score: {challenger_score}")
        print(f"Champion score: {champion_score}")
        elapsed = time.time() - start
        print("Cell run time:", time.strftime("%H:%M:%S", time.gmtime(elapsed)))

        # if challenger improves upon champion, then replace champion with challenger.
        if challenger_score < champion_score:
            features_champion += [feature_marginal_challenger]
            champion_score = challenger_score
            features_marginal.remove(feature_marginal_challenger)
            print(f"{feature_marginal_challenger} selected in this step.")

        champions_score_sequence.append(champion_score)

In [86]:
# ultimate bottleneck: in each stepwise 'round', considering ~500 features.
# evaluate features in parallel
from joblib import Parallel, delayed
import warnings

# it's strange to suppress a warning, but empirically and from research,
# haven't found the call-to-action from this warning.
warnings.filterwarnings("ignore", message=".*A worker stopped while some jobs were given to the executor.*", category=UserWarning)

start = time.time()

def fit_evaluate_marginal_model(
    feature_marginal, 
    # features_champion, 
    X_challenger, 
    y, 
    n_jobs):

    # features_challenger = features_champion + [feature_marginal]
    # X_challenger = X[features_challenger]

    features_challenger = list(X_challenger.columns)
    transformers = create_transformers_argument(features_challenger, COLUMNS_CATEGORICAL, COLUMNS_NUMERIC)
    feature_transform_pipeline = ColumnTransformer(transformers, remainder='passthrough', verbose_feature_names_out=False)
    pipeline_e2e = Pipeline(
        [
            ("transform_features", feature_transform_pipeline), 
            ("standard_scale", StandardScaler()),
            ('to_float32', Float32Transformer()),
            ("model", Ridge(alpha=0.1))
        ]
        )

    scores = cross_val_score(
        pipeline_e2e, 
        X_challenger, 
        y.astype('float32'), 
        scoring='neg_root_mean_squared_error', 
        cv=5,  
        n_jobs=n_jobs
        )
    score_cv_summary = scores.mean()

    return (feature_marginal, -1 * score_cv_summary)


if IS_FORWARD_STEPWISE_COMPLETE_FEATURE_SELECTION_RUN:

    features_marginal = features_categorical + features_onehot + features_numeric_impute
    # using features universe, model search times practically infeasible
    features_marginal = [x for x in features_marginal if x not in features_low_importance]
    features_champion = []
    features_added_count = 0

    # dummy, to kick off procedure
    champion_score = 10_000
    challenger_score = 1_000
    champions_score_sequence = [champion_score]

    # when challenger improves upon champion, continue extending challenger.
    # when challenger loses to champion, challenger has become too complex.
    while champion_score >= challenger_score:

        challengers_scores = Parallel(n_jobs=7)(
            delayed(fit_evaluate_marginal_model)(x, XY[features_champion + [x]], XY['listening_time_minutes'], 1) 
            for x in features_marginal
        )

        feature_marginal_challenger, challenger_score = min(challengers_scores, key=lambda x: x[1])

        print(f"Challenger score: {challenger_score}")
        print(f"Champion score: {champion_score}")
        elapsed = time.time() - start
        print("Cell run time:", time.strftime("%H:%M:%S", time.gmtime(elapsed)))

        # if challenger improves upon champion, then replace champion with challenger.
        if challenger_score < champion_score:

            features_champion += [feature_marginal_challenger]
            champion_score = challenger_score
            features_marginal.remove(feature_marginal_challenger)
            print(f"{feature_marginal_challenger} selected in this step.")

            features_added_count += 1
            print(f"Model features count comes to {features_added_count}.")

        champions_score_sequence.append(champion_score)

Challenger score: 13.543498039245605
Champion score: 10000
Cell run time: 00:00:12
episode_length_minutes selected in this step.
Model features count comes to 1.
Challenger score: 13.395149040222169
Champion score: 13.543498039245605
Cell run time: 00:00:47
number_of_ads selected in this step.
Model features count comes to 2.
Challenger score: 13.369787979125977
Champion score: 13.395149040222169
Cell run time: 00:01:18
host_popularity_percentage selected in this step.
Model features count comes to 3.
Challenger score: 13.359157752990722
Champion score: 13.369787979125977
Cell run time: 00:01:58
podcast_name_episode_sentiment selected in this step.
Model features count comes to 4.
Challenger score: 13.355846786499024
Champion score: 13.359157752990722
Cell run time: 00:03:41
episode_sentiment_Positive selected in this step.
Model features count comes to 5.
Challenger score: 13.353748512268066
Champion score: 13.355846786499024
Cell run time: 00:05:28
guest_popularity_percentage selecte

In [87]:
# forward stepwise features selection is time-consuming, so persist a checkpoint
if IS_FORWARD_STEPWISE_COMPLETE_FEATURE_SELECTION_RUN:

    with open('./models/features_forward_stepwise_selected.json', 'w') as file:
        json.dump(features_champion, file)
        FEATURES_LISTENING_TIME_MINUTES_MODEL = features_champion

else:
    with open('./models/features_forward_stepwise_selected.json', 'r') as file:
        FEATURES_LISTENING_TIME_MINUTES_MODEL = json.load(file)
    # with open('./models/features_forward_stepwise_selected_manual.txt', 'r') as file:
    #     FEATURES_LISTENING_TIME_MINUTES_MODEL = [line.strip() for line in file]

In [94]:
with open('./models/features_forward_stepwise_selected_baseline.json', 'r') as file:
    FEATURES_LISTENING_TIME_MINUTES_MODEL_BASELINE = json.load(file)

In [98]:
# len(set(FEATURES_LISTENING_TIME_MINUTES_MODEL_BASELINE) & set(FEATURES_LISTENING_TIME_MINUTES_MODEL))
set(FEATURES_LISTENING_TIME_MINUTES_MODEL_BASELINE).difference(set(FEATURES_LISTENING_TIME_MINUTES_MODEL))

{'episode_count',
 'genre_Education',
 'podcast_name',
 'podcast_name_Business Briefs',
 'podcast_name_Comedy Corner',
 'podcast_name_Finance Focus',
 'podcast_name_Game Day',
 'podcast_name_Health Hour',
 'podcast_name_Innovators',
 'podcast_name_Laugh Line',
 'podcast_name_Music Matters',
 'podcast_name_True Crime Stories',
 'podcast_name_Tune Time',
 'publication_day_Saturday',
 'publication_day_Sunday',
 'publication_day_Tuesday',
 'publication_time',
 'publication_time_Morning'}

In [29]:
# if IS_FORWARD_STEPWISE_EPISODE_LENGTH_IMPUTE_MODEL_FEATURE_SELECTION_RUN:

#     XY_cc = XY.dropna(subset=['episode_length_minutes'])

#     features_marginal = features_categorical + features_onehot + features_numeric_impute
#     features_marginal.remove('episode_length_minutes')
#     features_champion = []

#     champion_score = 10_000
#     challenger_score = 1_000
#     champions_score_sequence = [champion_score]

#     # when challenger improves upon champion, continue extending challenger.
#     # when challenger loses to champion, challenger has become too complex.
#     while champion_score >= challenger_score:

#         challengers_scores = {}
#         for feature_marginal in features_marginal:

#             features_challenger = features_champion + [feature_marginal]

#             X_challenger = XY_cc[features_challenger]
#             y = XY_cc['episode_length_minutes']

#             transformers = create_transformers_argument(features_challenger, features_categorical, features_numeric_impute)
#             feature_transform_pipeline = ColumnTransformer(transformers, remainder='passthrough', verbose_feature_names_out=False)

#             pipeline_e2e = Pipeline(
#                 [
#                     ("transform_features", feature_transform_pipeline), 
#                     ("model", Ridge(alpha=0.1))
#                 ]
#                 )
            
#             scores = cross_val_score(
#                 pipeline_e2e, 
#                 X_challenger, 
#                 y, 
#                 scoring='neg_root_mean_squared_error', 
#                 cv=5,  
#                 n_jobs=-1
#                 )
#             score_cv_summary = scores.mean()

#             challengers_scores[feature_marginal] = -1 * score_cv_summary

#         feature_marginal_challenger = min(challengers_scores, key=challengers_scores.get)
#         challenger_score = challengers_scores[feature_marginal_challenger]

#         print(f"Challenger score: {challenger_score}")
#         print(f"Champion score: {champion_score}")

#         # if challenger improves upon champion, then replace champion with challenger.
#         if challenger_score < champion_score:
#             features_champion += [feature_marginal_challenger]
#             champion_score = challenger_score
#             features_marginal.remove(feature_marginal_challenger)
#             print(f"{feature_marginal_challenger} selected in this step.")

#         champions_score_sequence.append(champion_score)

In [30]:
# if IS_FORWARD_STEPWISE_EPISODE_LENGTH_IMPUTE_MODEL_FEATURE_SELECTION_RUN:

#     with open('./models/features_model_episode_length_minutes.json', 'w') as file:
#         json.dump(features_champion, file)
#         FEATURES_EPISODE_LENGTH_MINUTES_MODEL = features_champion

# else:
#     with open('./models/features_model_episode_length_minutes.json', 'r') as file:
#         FEATURES_EPISODE_LENGTH_MINUTES_MODEL = json.load(file)

# Stronger Fit Model

## Test Simple Imputations

In [114]:
# FEATURES_LISTENING_TIME_MINUTES_MODEL
list(
    # set(FEATURES_LISTENING_TIME_MINUTES_MODEL) - set(FEATURES_LISTENING_TIME_MINUTES_MODEL_BASELINE)
    set(FEATURES_LISTENING_TIME_MINUTES_MODEL_BASELINE) - set(FEATURES_LISTENING_TIME_MINUTES_MODEL)
    )

['genre_Education',
 'podcast_name_True Crime Stories',
 'podcast_name_Game Day',
 'podcast_name_Finance Focus',
 'podcast_name',
 'podcast_name_Innovators',
 'podcast_name_Tune Time',
 'publication_day_Saturday',
 'publication_day_Tuesday',
 'publication_time_Morning',
 'publication_time',
 'podcast_name_Business Briefs',
 'podcast_name_Music Matters',
 'publication_day_Sunday',
 'episode_count',
 'podcast_name_Comedy Corner',
 'podcast_name_Health Hour',
 'podcast_name_Laugh Line']

In [115]:
features_low_importance

['podcast_name_publication_day_Style Guide_Saturday',
 'podcast_name_episode_sentiment_Health Hour_Negative',
 'podcast_name_publication_day_Funny Folks_Monday',
 'podcast_name_publication_time_Humor Hub_Afternoon',
 'podcast_name_publication_time_Finance Focus_Evening',
 'podcast_name_publication_time_Healthy Living_Evening',
 'podcast_name_publication_time_Fashion Forward_Evening',
 'podcast_name_episode_sentiment_Gadget Geek_Positive',
 'genre_publication_day_Education_Thursday',
 'genre_publication_day_Business_Thursday',
 'podcast_name_publication_time_Tech Talks_Evening',
 'podcast_name_publication_time_Laugh Line_Morning',
 'podcast_name_publication_time_Joke Junction_Afternoon',
 'podcast_name_episode_sentiment_Gadget Geek_Negative',
 'podcast_name_episode_sentiment_Detective Diaries_Neutral',
 'podcast_name_publication_day_Style Guide_Monday',
 'podcast_name_episode_sentiment_Mind & Body_Positive',
 'podcast_name_publication_day_Funny Folks_Friday',
 'podcast_name_publication_

In [102]:
FEATURES_LISTENING_TIME_MINUTES_MODEL_COMMON = list(
    set(FEATURES_LISTENING_TIME_MINUTES_MODEL) & set(FEATURES_LISTENING_TIME_MINUTES_MODEL_BASELINE)
    )

FEATURES_LISTENING_TIME_MINUTES_MODEL_UNION = list(
    set(FEATURES_LISTENING_TIME_MINUTES_MODEL) | set(FEATURES_LISTENING_TIME_MINUTES_MODEL_BASELINE)
)

In [111]:
transformers = []

# features_challenger_target_encode = [ftr for ftr in FEATURES_LISTENING_TIME_MINUTES_MODEL_UNION if ftr in COLUMNS_CATEGORICAL]
features_challenger_target_encode = COLUMNS_CATEGORICAL
if features_challenger_target_encode:
    spec = ('target_encode', TargetEncoder(), features_challenger_target_encode)
    transformers.append(spec)

# features_challenger_impute = [ftr for ftr in FEATURES_LISTENING_TIME_MINUTES_MODEL_UNION if ftr in COLUMNS_NUMERIC]
features_challenger_impute = COLUMNS_NUMERIC
if features_challenger_impute:
    spec = ('impute', SimpleImputer(strategy='mean', add_indicator=True), features_challenger_impute)

    transformers.append(spec)

feature_transform_pipeline = ColumnTransformer(
    transformers, 
    remainder='passthrough', 
    verbose_feature_names_out=False
    ).set_output(transform='pandas')

pipeline_e2e = Pipeline(
    [
        ("transform_features", feature_transform_pipeline), 
        # CAUTION: 1.0 instructs, sample entire features universe. 1 (integer) doesn't instruct the same.
        ("model", RandomForestRegressor(100, max_features=1.0, n_jobs=-1))
    ]
    )

scores = cross_val_score(
    pipeline_e2e, 
    
    XY[COLUMNS_CATEGORICAL + COLUMNS_NUMERIC],
    # XY[FEATURES_LISTENING_TIME_MINUTES_MODEL_UNION], 
    # XY[features_challenger_target_encode + features_challenger_impute + features_onehot],
    
    XY['listening_time_minutes'], 
    
    scoring='neg_root_mean_squared_error', 
    
    cv=5
    )
score_cv_summary = scores.mean()

score_cv_summary

np.float64(-13.139493580162389)

In [ ]:
# the more features i declare for Target Encoding, the worse the score becomes ...
# does this indicate TargetEncoder data leakage under the hood?

## Learn from Weak Signals with Optimized GBM

In [108]:
# optimize gradient boosting machine hyperparams,
# according to out-of-fold average loss.

# if xgboost model can inject to sklearn pipeline abstraction,
# then objective score is simply cross_val_score.

# initial xgboost train scored 13 -- not better than preceding RF.

# Kaggle Grandmaster observes gradient boosting's merit for this competition:
# https://www.kaggle.com/competitions/playground-series-s5e4/discussion/571549

import optuna
import xgboost as xgb

transformers = []

# features_challenger_target_encode = [ftr for ftr in FEATURES_LISTENING_TIME_MINUTES_MODEL if ftr in COLUMNS_CATEGORICAL]
features_challenger_target_encode = COLUMNS_CATEGORICAL
if features_challenger_target_encode:
    spec = ('target_encode', TargetEncoder(), features_challenger_target_encode)
    transformers.append(spec)

# features_challenger_impute = [ftr for ftr in FEATURES_LISTENING_TIME_MINUTES_MODEL if ftr in COLUMNS_NUMERIC]
features_challenger_impute = COLUMNS_NUMERIC
if features_challenger_impute:
    spec = ('impute', SimpleImputer(strategy='mean', add_indicator=True), features_challenger_impute)
    transformers.append(spec)

feature_transform_pipeline = ColumnTransformer(
    transformers, 
    remainder='passthrough', 
    verbose_feature_names_out=False
    ).set_output(transform='pandas')

def objective(trial):

    params = {
        "n_estimators": 1_000,
        "objective": "reg:squarederror",
        "booster": "gbtree",
        # approximately sorted on hyperparam influence, descending
        "eta": trial.suggest_float("eta", 1e-5, 0.01, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 9, step=2),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0),
        "lambda": trial.suggest_float("lambda", 1e-8, 1.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        }
    
    pipeline_e2e = Pipeline(
    [
        ("transform_features", feature_transform_pipeline), 
        ("model", xgb.XGBRegressor(**params))
    ]
    )

    scores = cross_val_score(
        pipeline_e2e, 

        XY[COLUMNS_CATEGORICAL + COLUMNS_NUMERIC],
        # XY[FEATURES_LISTENING_TIME_MINUTES_MODEL],

        XY['listening_time_minutes'], 

        # as optuna _minimizes_, beware initially negative result
        scoring='neg_root_mean_squared_error', 

        cv=5
        )
    score_cv_summary = scores.mean()

    return -1 * score_cv_summary

study = optuna.create_study()

study.optimize(
    objective,
    # as hyperparameters set size increases, likely need more trials
    n_trials=25,
    # if optuna returns nulls in y_pred, don't fail the entire study
    catch=(ValueError,),
    n_jobs=-1,
    timeout=1 * 60 * 60,
)
    

[I 2025-04-11 09:21:06,374] A new study created in memory with name: no-name-ee1374ee-935b-42f7-a280-525352208154
[I 2025-04-11 09:23:13,546] Trial 6 finished with value: 14.265474346671027 and parameters: {'eta': 0.004389533750954217, 'max_depth': 1, 'subsample': 0.8900742513367644, 'colsample_bytree': 0.7031806319087193, 'lambda': 3.7727176869604763e-07, 'alpha': 0.4808115851265626, 'gamma': 2.5474061464246926e-06}. Best is trial 6 with value: 14.265474346671027.
[I 2025-04-11 09:23:14,913] Trial 4 finished with value: 26.943981870067056 and parameters: {'eta': 1.9680501573680306e-05, 'max_depth': 1, 'subsample': 0.9481083249877377, 'colsample_bytree': 0.7229306152742733, 'lambda': 1.1177274455149722e-08, 'alpha': 0.343054712268586, 'gamma': 0.3029394293427033}. Best is trial 6 with value: 14.265474346671027.
[I 2025-04-11 09:23:16,769] Trial 2 finished with value: 25.04883339400767 and parameters: {'eta': 0.0003303043412522784, 'max_depth': 1, 'subsample': 0.6582171788891972, 'colsa

## Test Complex Imputations

In [31]:
class GroupImputer(BaseEstimator, TransformerMixin):
    """"
    When a column's values differ significantly by group,
    prefer an imputer which fits & transforms mean by group.
    Transformer operates over several columns of values.
    Allow groupby column drop, to avoid name conflicts with other columns during Pipeline concatenations.
    """
    def __init__(
        self, 
        column_groupby: str, 
        columns_values_to_impute: list, 
        strategy='mean', 
        drop_groupby_transform=True
        ):
        
        self.column_groupby = column_groupby
        self.columns_values_to_impute = columns_values_to_impute
        self.strategy = strategy
        self.imputer = SimpleImputer(strategy=self.strategy)
        self.drop_groupby_transform = drop_groupby_transform
        
    def fit(self, X, y=None):

        self.groups_imputers_ = {}
        
        for group_title, observations in X.groupby(self.column_groupby):

            self.imputer.fit(observations[self.columns_values_to_impute])

            self.groups_imputers_[group_title] = self.imputer
        
        return self
    
    def transform(self, X):

        # client doesn't expect original data to be modified
        X_transformed = X.copy()

        for col in self.columns_values_to_impute:
            X_transformed[f"missingindicator_{col}"] = X_transformed[col].isnull().astype(int)
        
        for group_title, observations in X_transformed.groupby(self.column_groupby):

            X_transformed.loc[observations.index, self.columns_values_to_impute] = (
                self.groups_imputers_[group_title]
                .transform(observations[self.columns_values_to_impute])
                )
            
        if self.drop_groupby_transform:
            X_transformed = X_transformed.drop(columns=self.column_groupby)
        
        return X_transformed
    
    def set_output(self, transform='pandas'):
        return self

In [78]:
class CustomImputePredictPipeline(BaseEstimator, TransformerMixin):
    """
    Fit two supervised models:
        2. Model of ultimate outcome `listening_time_minutes`, which depends on
        1. Model of strongly influential feature `episode_length_minutes`

    Each step needs a dedicated feature transform pipeline -- cannot share between steps --
    because target encoding result differs by outcome. 
    Either feature transform pipeline generally involves target encoding and simple imputations.

    Because pipeline involves two different outcomes -- imputation supervisor, ultimate `y` -- 
    bespoke Pipeline required, off-the-shelf isn't flexible enough.

    """

    def __init__(self, hyperparams_ultimate_model=None):

        self.features_target_encode = [
            x for x in FEATURES_LISTENING_TIME_MINUTES_MODEL if x in COLUMNS_CATEGORICAL
            ]
        
        self.features_numeric_simple_impute = [
            x for x in FEATURES_LISTENING_TIME_MINUTES_MODEL if x in COLUMNS_NUMERIC
            ]
        self.features_numeric_simple_impute.remove('episode_length_minutes')
        self.hyperparams_ultimate_model = hyperparams_ultimate_model

        # from preceding feature selection
        self.features_impute_episode_length_minutes = [
            'number_of_ads',
            'podcast_name',
            'episode_sentiment_Negative',
            'host_popularity_percentage',
            'publication_time',
            'publication_day',
            'episode_count',
            'guest_popularity_percentage',
            'podcast_name_Home & Living'
            ]
        self.features_imputer_model_target_encode = [
            x for x in self.features_impute_episode_length_minutes if x in COLUMNS_CATEGORICAL
            ]
        self.features_imputer_model_simple_impute = [
            x for x in self.features_impute_episode_length_minutes if x in COLUMNS_NUMERIC
            ]

    def fit(self, X, y=None):
        """Ultimate `y` of interest."""

        self.fit_imputer_model(X)

        self.fit_ultimate_model(X, y)


    def fit_imputer_model(self, X):

        X_impute = (
            X
            .dropna(subset='episode_length_minutes')
            [self.features_impute_episode_length_minutes]
            .copy()
            )
        y_impute = X['episode_length_minutes'].dropna().copy()

        imputer_feature_transform_pipeline = ColumnTransformer(
            [
                ('target_encode', TargetEncoder(), self.features_imputer_model_target_encode),
                ('impute', SimpleImputer(strategy='mean', add_indicator=True), self.features_imputer_model_simple_impute)
            ], 
            # safe because, explicit columns select follows a couple lines above
            remainder='passthrough', 
            verbose_feature_names_out=False
            ).set_output(transform='pandas')
        
        imputer_pipeline_e2e = Pipeline(
            [
                ("transform_features", imputer_feature_transform_pipeline), 
                ("model", Ridge(alpha=0.1))
            ]
            )
        
        imputer_pipeline_e2e.fit(X_impute, y_impute)
        self.imputer_pipeline_e2e = imputer_pipeline_e2e

    def transform_imputer_model(self, X):

        X_trfm = X.copy()

        has_null_impute_target = X_trfm['episode_length_minutes'].isnull()
        X_trfm.loc[has_null_impute_target, 'episode_length_minutes'] = (
            self.imputer_pipeline_e2e.predict(X_trfm.loc[has_null_impute_target])
        )
        
        X_trfm["missingindicator_episode_length_minutes"] = has_null_impute_target.astype(int)

        return X_trfm


    def fit_ultimate_model(self, X, y):

        X_trfm = self.transform_imputer_model(X)

        X_trfm = X_trfm[FEATURES_LISTENING_TIME_MINUTES_MODEL + ['missingindicator_episode_length_minutes']]
        
        feature_transform_pipeline = ColumnTransformer(
            [
                ('target_encode', TargetEncoder(), self.features_target_encode),
                ('impute', SimpleImputer(strategy='mean', add_indicator=True), self.features_numeric_simple_impute)
            ], 
            # safe because, explicit columns select follows a couple lines above
            remainder='passthrough', 
            verbose_feature_names_out=False
            ).set_output(transform='pandas')
        
        ultimate_pipeline_e2e = Pipeline(
            [
                ("transform_features", feature_transform_pipeline), 
                ("model", RandomForestRegressor(**self.hyperparams_ultimate_model))
            ]
            )
        
        ultimate_pipeline_e2e.fit(X_trfm, y)
        self.ultimate_pipeline_e2e = ultimate_pipeline_e2e

    def predict(self, X):

        X_trfm = self.transform_imputer_model(X)
        preds = self.ultimate_pipeline_e2e.predict(X_trfm)

        return preds

    def set_output(self, transform='pandas'):
        return self

In [79]:
# # confirm that imputed values vary
# X_test_impute = XY.copy()

# extended_pipeline = CustomImputePredictPipeline()
# extended_pipeline.fit_imputer_model(X_test_impute)
# X_test_impute = extended_pipeline.transform_imputer_model(X_test_impute)

# # *low* variation among imputed values
# X_test_impute.groupby('missingindicator_episode_length_minutes')['episode_length_minutes'].describe()

# # (
# #     p9.ggplot(X_test_impute) + 
# #     p9.theme_bw() + 
# #     p9.geom_density(
# #         p9.aes('episode_length_minutes', group='missingindicator_episode_length_minutes', color='missingindicator_episode_length_minutes')
# #         )
# # )

In [ ]:
# extended_pipeline = CustomImputePredictPipeline()
# scores = cross_val_score(
#     extended_pipeline, 
#     XY.drop(columns='listening_time_minutes'), 
#     XY['listening_time_minutes'], 
#     scoring='neg_root_mean_squared_error', 
#     cv=5,  
#     n_jobs=-1
#     )
# score_cv_summary = scores.mean()
# score_cv_summary

kf = KFold(n_splits=5, shuffle=True, random_state=777)

predictions_inspect = []

for indexes_train, indexes_test in kf.split(XY):

    X_train = XY.drop(columns='listening_time_minutes').iloc[indexes_train]
    y_train = XY['listening_time_minutes'].iloc[indexes_train]

    X_test = XY.drop(columns='listening_time_minutes').iloc[indexes_test]
    y_test = XY['listening_time_minutes'].iloc[indexes_test]
    
    pipeline_e2e = CustomImputePredictPipeline(hyperparams_ultimate_model={"n_jobs": -1})
    pipeline_e2e.fit(X_train, y_train)

    predictions = pipeline_e2e.predict(X_test)

    predictions = (
        X_test
        .copy()
        .assign(
            y = y_test, 
            pred = predictions
            )
        .assign(
            error = lambda df_: df_['y'] - df_['pred'],
            error2 = lambda df_: (df_['y'] - df_['pred'])**2
            )
        )

    predictions_inspect.append(predictions)

    print("cv fold complete.")

predictions_inspect = pd.concat(predictions_inspect, axis=0)

In [ ]:
root_mean_squared_error(
    predictions_inspect['y'], 
    predictions_inspect['pred']
    )

In [85]:
predictions_inspect.to_csv("./data/processed/predictions_inspect_submission1.csv", index=False)

In [ ]:
# to improve model errors,
    # utilize current model features better (transforms, revised model structure)
    # incorporate new information (features that have been excluded, etc)

imputer_adhoc = ColumnTransformer(
    [('impute', SimpleImputer(strategy='mean', add_indicator=True), COLUMNS_NUMERIC)],
    remainder='passthrough',
    verbose_feature_names_out=False
    ).set_output(transform='pandas')

predictions_inspect_cc = imputer_adhoc.fit_transform(predictions_inspect)

predictions_inspect_cc = predictions_inspect_cc.sample(100_000)

mutual_information = mutual_info_regression(
    predictions_inspect_cc[COLUMNS_NUMERIC + features_onehot], 
    predictions_inspect_cc['error']
    )

mutual_information = (
    pd.Series(mutual_information, index=COLUMNS_NUMERIC + features_onehot)
    .sort_values(ascending=False)
)
mutual_information

In [ ]:
predictions_inspect_cc.plot.scatter('episode_length_minutes', 'error');

In [ ]:
# # from predictions inspection: significant performance variation by podcast 
# # then, hypothesis: fit by podcast_name

# podcasts_predictions = []

# # for each podcast_name [segment]: (generator object -- only need 1 segment at a time)
# for podcast_name, XY_grp in XY.groupby('podcast_name'):
    
#     # per cv fold, fit model & predict test
#     kf = KFold(n_splits=5, shuffle=True, random_state=777)
#     predictions_podcast = []
#     for indexes_train, indexes_test in kf.split(XY_grp):
            
#         X_train = XY_grp[features_champion].iloc[indexes_train]
#         y_train = XY_grp['listening_time_minutes'].iloc[indexes_train]

#         X_test = XY_grp[features_champion].iloc[indexes_test]
#         y_test = XY_grp['listening_time_minutes'].iloc[indexes_test]

#         pipeline_e2e = Pipeline(
#             [
#                 ("transform_features", feature_transform_pipeline), 
#                 ("model", RandomForestRegressor(n_jobs=-1))
#             ]
#             )
        
#         pipeline_e2e.fit(X_train, y_train)

#         predictions = pipeline_e2e.predict(X_test)
#         predictions_enriched = (
#             XY_grp
#             [['id', 'podcast_name', 'listening_time_minutes']]
#             .iloc[indexes_test]
#             .assign(pred = predictions)
#         )
        
#         predictions_podcast.append(predictions_enriched)

#         print("cv fold complete.")

#     # combine podcast's test predictions 
#     predictions_podcast = pd.concat(predictions_podcast, axis=0)

#     podcasts_predictions.append(predictions_podcast)

#     print(f"Predictions complete: {podcast_name}")

# # combine full set's test predictions
# podcasts_predictions = pd.concat(podcasts_predictions, axis=0)

# # score full set
# root_mean_squared_error(
#     podcasts_predictions['listening_time_minutes'], 
#     podcasts_predictions['pred']
#     )

# Inspect Predictions

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=777)

predictions_inspect = []

for indexes_train, indexes_test in kf.split(XY):

    X_train = XY[FEATURES_LISTENING_TIME_MINUTES_MODEL].iloc[indexes_train]
    y_train = XY['listening_time_minutes'].iloc[indexes_train]

    X_test = X_challenger.iloc[indexes_test]
    y_test = y.iloc[indexes_test]

    pipeline_e2e = Pipeline(
        [
            ("transform_features", feature_transform_pipeline), 
            ("model", RandomForestRegressor(n_jobs=-1))
        ]
        )
    
    pipeline_e2e.fit(X_train, y_train)

    predictions = pipeline_e2e.predict(X_test)

    predictions = (
        X_test
        .copy()
        .assign(y = y_test, pred = predictions)
        .assign(error2 = lambda df_: (df_['y'] - df_['pred'])**2)
    )

    predictions_inspect.append(predictions)

    print("cv fold complete.")

predictions_inspect = pd.concat(predictions_inspect, axis=0)

In [ ]:
root_mean_squared_error(
    predictions_inspect['y'], 
    predictions_inspect['pred']
    )

In [61]:
predictions_inspect = predictions_inspect.assign(
    pred_bucket = lambda df_: pd.qcut(df_['pred'], q=5, precision=2)
    )

predictions_inspect.to_csv("./data/processed/predictions_inspect.csv", index=False)

Observations:

- Highest error^2 often see listening_time_minutes >> episode_length_minutes. That's unusual, as 90% of observations have listening_time_minutes <= episode_length_minutes. 

- Highest error^2 more frequently have missing episode_length_minutes, an important feature.

- RMSE varies by podcast_name. Top 2 data segments by observation count--Tech Talks, Sports Weekly--have RMSE of 12.7. In top 5 segments, some RMSE score 12.4 or 12.5. _Forward-stepwise feature selection won't detect significantly different interactions by podcast_name. Moreover, Sports Weekly didn't even pass feature selection screen.
    - Hypothesis: to capture interaction relationships more fully, fit model by data segment.


# Train "Production" Models

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=777)

models_production = []

for indexes_train, indexes_test in kf.split(X, y):

    X_train = X_challenger.iloc[indexes_train]
    y_train = y.iloc[indexes_train]

    pipeline_e2e = Pipeline(
        [
            ("transform_features", feature_transform_pipeline), 
            ("model", RandomForestRegressor(n_jobs=-1))
        ]
        )
    
    pipeline_e2e.fit(X_train, y_train)

    print(f"number_of_ads null rate: {X_train['number_of_ads'].isnull().mean()}")
    print("Model fit complete.")

    models_production.append(pipeline_e2e)

In [ ]:
# TODO: number_of_ads 0% null rate raises issues on predict. 
# column_transformer suggests output of missingindicator,
# but RandomForestRegressor suggests that indicator doesn't exist.
# because this by-cv model is a nice-to-have, not imperative -- drop.
models_production.pop(2)

In [ ]:
# test method of multi-models prediction, then aggregation.
# expected result: error score is optimistic versus out-of-fold estimates

# predictions_models_production = [model.predict(X_challenger) for model in models_production]

# predictions_integrated = np.column_stack(predictions_models_production)
# predictions_agg = predictions_integrated.mean(axis=1)

# root_mean_squared_error(y, predictions_agg)

# Deploy

In [133]:
podcasts_test = pd.read_csv("./data/external/test.csv")
podcasts_test.columns = [x.lower() for x in podcasts_test.columns]

podcasts_test = podcasts_test.assign(
    episode_count = lambda df_: df_['episode_title'].str.extract(r'Episode (\d+)').astype(int)
)

X_onehot = onehot_encoder.transform(podcasts_test[features_categorical])
features_onehot = list(X_onehot.columns)
podcasts_test = pd.concat([podcasts_test, X_onehot], axis=1)

In [138]:
predictions_models_suite_podcasts_test = [
    model.predict(podcasts_test) 
    for model in models_production
    ]

predictions_integrated = np.column_stack(predictions_models_suite_podcasts_test)
predictions_agg = predictions_integrated.mean(axis=1)

assert podcasts_test.shape[0] == predictions_agg.shape[0]

In [140]:
predictions_test = podcasts_test[['id']].assign(Listening_Time_minutes = predictions_agg)
predictions_test.to_csv("./data/processed/submission1.csv", index=False)